# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**:

**ID**:

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/BEE 4750 /HW 5/hw5-parker-regina-1`
   Installed PlotUtils ────────── v1.4.4
   Installed GR_jll ───────────── v0.73.18+0
   Installed OpenBLAS32_jll ───── v0.3.29+0
   Installed Measures ─────────── v0.3.3
   Installed OpenSSL ──────────── v1.6.0
   Installed StaticArrays ─────── v1.9.15
   Installed MutableArithmetics ─ v1.6.7
   Installed HiGHS_jll ────────── v1.12.0+0
   Installed Pango_jll ────────── v1.57.0+0
   Installed JSON ─────────────── v1.3.0
   Installed FFMPEG ───────────── v0.4.5
   Installed StaticArraysCore ─── v1.4.4
   Installed GR ───────────────── v0.73.18
   Installed DataStructures ───── v0.19.3
   Installed METIS_jll ────────── v5.1.3+0
   Installed Graphs ───────────── v1.13.1
   Installed GraphRecipes ─────── v0.5.15
   Installed StatsBase ────────── v0.34.8
   Installed ChainRulesCore ───── v1.26.0
   Installed SimpleTraits ─────── v0.9.5
   Installed HiGHS ────────────── v1.20.1
   Installed JuMP ─────────────── v1.29.3
   Install

In [3]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

In [2]:
ash_frac = (.15*.08) + (.40*.03) + (.05*0.05) + (.10*0.02) + (.02*0.15) + (.05*0.02) + (.18*0.02) + (.04*1.00) + (.05*1.00) + (.02*1.00) + (.01*1.00) + (.03*0.70)
recycle_frac = (.15*0) + (.40*.55) + (.05*0.15) + (.10*0) + (.02*0) + (.05*0.30) + (.18*0.40) + (.04*0.60) + (.05*0.75) + (.02*0.80) + (.01*0.50) + (.03*0)

println("Recycling fraction = ", round(recycle_frac; digits=3))
println("Ash fraction = ", round(ash_frac; digits=3))

Recycling fraction = 0.397
Ash fraction = 0.177


**Solution 1.1**
Because all three cities have the same exact waste composition table, their overall recycling and ash fractions will be the same. The recycling fraction was found by finding the product of each component's % of total mass and MRF recycling rate and then summing this number for all components. The ash fraction was found the same way except using the % of combustion ash instead of the MRF recycling rate. The results yielded a $17.7$% ash fraction and $39.7$% recycling fraction.

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

**Solution 1.2**

The decision variables are:

- $x_i^{L}$ : mass of MSW (Mg/day) transported from city $i$ to the landfill (LF).
- $x_i^{M}$ : mass of MSW (Mg/day) transported from city $i$ to the materials recovery facility (MRF).
- $x_i^{W}$ : mass of MSW (Mg/day) transported from city $i$ to the waste-to-energy facility (WTE).

Binary facility-activation variables:

- $y_{L} = 1$ if the landfill is operating, and $0$ otherwise.
- $y_{M} = 1$ if the MRF is operating, and $0$ otherwise.
- $y_{W} = 1$ if the WTE is operating, and $0$ otherwise.

Parameters from Problem 1.1 (used in the objective and constraints):

- $r$ : overall recycling fraction at the MRF, $r \approx 0.397$.
- $a$ : overall ash fraction for waste sent to WTE, $a \approx 0.177$.


#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

**Solution 1.3**

Thus our final mixed-integer linear program is:

$\min_{x_i^{L}, x_i^{M}, x_i^{W}, y_L, y_M, y_W}
\Bigg(
\sum_i \left[ 1.5\,d_{iL}\,x_i^{L} + 1.5\,d_{iM}\,x_i^{M} + 1.5\,d_{iW}\,x_i^{W}
+ 50\,x_i^{L} + (7 + 40\,r)\,x_i^{M} + 60\,x_i^{W} \right]
+ 2000\,y_L + 1500\,y_M + 2500\,y_W
\Bigg)$

where:

- $d_{iL}, d_{iM}, d_{iW}$ are distances (km) from city $i$ to the landfill, MRF, and WTE.
- $1.5$ is the transportation cost in dollars per Mg-km.
- $50$, $7$, and $60$ are the tipping/processing costs in dollars per Mg at LF, MRF, and WTE.
- $40$ is the additional cost in dollars per Mg of material that is recycled at the MRF.
- $r$ is the overall recycling fraction (from Problem 1.1), so the effective MRF cost per Mg is $(7 + 40r)$.
- $2000$, $1500$, and $2500$ are the fixed daily operating costs for LF, MRF, and WTE.

The objective function minimizes the total daily cost of handling and processing all municipal solid waste generated by the three cities. The total cost has three main components:

1. **Transportation Costs**  
   Waste shipped from each city $i$ to each facility (LF, MRF, WTE) incurs a transportation cost of $1.5$ dollars per Mg-km.  
   This produces the transport cost term  
   $1.5\,d_{iL}\,x_i^{L} + 1.5\,d_{iM}\,x_i^{M} + 1.5\,d_{iW}\,x_i^{W}$  
   for each city $i$.

2. **Processing / Tipping Costs**  
   Each facility charges a per-Mg fee for incoming waste:  
   - Landfill: $50$ dollars/Mg  
   - MRF: $7$ dollars/Mg  
   - WTE: $60$ dollars/Mg  

   The MRF also requires an extra $40$ dollars for every Mg of *recycled* material.  
   With the overall recycling fraction $r$ from Problem 1.1, the effective MRF cost per Mg is  
   $7 + 40r$.  
   Thus, the processing cost is  
   $50\,x_i^{L} + (7 + 40r)\,x_i^{M} + 60\,x_i^{W}$.

3. **Fixed Facility Operating Costs**  
   Each disposal facility has a fixed daily operating cost:  
   - Landfill: $2000$ dollars/day  
   - MRF: $1500$ dollars/day  
   - WTE: $2500$ dollars/day  

   These apply when the facility is used, captured through binary variables $y_L$, $y_M$, and $y_W$.  
   The fixed cost term is  
   $2000\,y_L + 1500\,y_M + 2500\,y_W$.

Combining transportation, processing, and fixed operating costs yields the total cost to be minimized:

$\sum_i \left[ 1.5\,d_{iL}\,x_i^{L} + 1.5\,d_{iM}\,x_i^{M} + 1.5\,d_{iW}\,x_i^{W} + 50\,x_i^{L} + (7 + 40r)\,x_i^{M} + 60\,x_i^{W} \right] \;+\; 2000\,y_L + 1500\,y_M + 2500\,y_W$




#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

**Solution 1.4**

The constraints for the optimization problem are:

1. **City mass-balance constraints**  
   All waste generated in each city must be sent to one of the three facilities:

   $x_i^{L} + x_i^{M} + x_i^{W} = Q_i$

   for each city $i$, where $Q_i$ is the MSW generated (Mg/day) in city $i$.

2. **Facility capacity constraints**

   - **Landfill capacity, including WTE ash:**

     $\sum_i x_i^{L} \;+\; a \sum_i x_i^{W} \;\le\; 200\,y_L$

   - **MRF capacity:**

     $\sum_i x_i^{M} \;\le\; 350\,y_M$

   - **WTE capacity:**

     $\sum_i x_i^{W} \;\le\; 210\,y_W$

   Here $a$ is the overall ash fraction from Problem 1.1.

3. **Nonnegativity and integrality**

   $x_i^{L} \ge 0,\quad x_i^{M} \ge 0,\quad x_i^{W} \ge 0$

   $y_L,\, y_M,\, y_W \in \{0,1\}$

The parameters $r$ and $a$ come from the waste composition table, with  
$r \approx 0.397$ (recycling fraction) and $a \approx 0.177$ (ash fraction).


#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

**Solution 1.5**

Combining the objective function and constraints from Solutions 1.2–1.4, the final mixed-integer program is:

$\min_{x_i^{L}, x_i^{M}, x_i^{W}, y_L, y_M, y_W}
\left[
\sum_i \left( 1.5\,d_{iL}\,x_i^{L} + 1.5\,d_{iM}\,x_i^{M} + 1.5\,d_{iW}\,x_i^{W}
+ 50\,x_i^{L} + (7 + 40r)\,x_i^{M} + 60\,x_i^{W} \right)
+ 2000\,y_L + 1500\,y_M + 2500\,y_W
\right]$

subject to:

City mass balance (all waste must be allocated):

$x_i^{L} + x_i^{M} + x_i^{W} = Q_i \quad \text{for each city } i$

Facility capacity constraints:

$\sum_i x_i^{L} + a \sum_i x_i^{W} \le 200\,y_L$

$\sum_i x_i^{M} \le 350\,y_M$

$\sum_i x_i^{W} \le 210\,y_W$

Nonnegativity and integrality:

$x_i^{L} \ge 0,\; x_i^{M} \ge 0,\; x_i^{W} \ge 0$

$y_L,\; y_M,\; y_W \in \{0,1\}$

Here, $r \approx 0.397$ is the overall recycling fraction and $a \approx 0.177$ is the ash fraction from Problem 1.1.

Using JuMP to solve this mixed-integer program, the optimal solution is:

- City 1: $x_{1}^{L} = 80$, $x_{1}^{M} = 20$, $x_{1}^{W} = 0$  
- City 2: $x_{2}^{L} = 0$, $x_{2}^{M} = 90$, $x_{2}^{W} = 0$  
- City 3: $x_{3}^{L} = 120$, $x_{3}^{M} = 0$, $x_{3}^{W} = 0$

Facility use:

- $y_{L} = 1$ (landfill open)  
- $y_{M} = 1$ (MRF open)  
- $y_{W} = 0$ (WTE closed)

The corresponding optimal objective value (total daily cost) is:

$Z^{*} = 23231.8 \;\text{dollars/day}$  

Rounded: $Z^{*} \approx 23232 \text{ dollars/day}$.


In [3]:
using JuMP
using HiGHS   # or GLPK, Gurobi, etc.


cities = 1:3

#daily MSW generation (Mg/day) for each city
Q = Dict(1 => 100.0,   # City 1
         2 => 90.0,    # City 2
         3 => 120.0)   # City 3

#distances (km) from each city to each facility
d_L = Dict(1 => 5.0,  2 => 15.0, 3 => 13.0)   # to Landfill
d_M = Dict(1 => 30.0, 2 => 25.0, 3 => 45.0)   # to MRF
d_W = Dict(1 => 15.0, 2 => 10.0, 3 => 20.0)   # to WTE

#transportation cost (dollars per Mg-km)
c_trans = 1.5

#tipping / processing costs (dollars per Mg of waste entering the facility)
c_L = 50.0    # landfill
c_M = 7.0     # MRF base tipping
c_W = 60.0    # WTE

#extra recycling cost at MRF: 40 dollars per Mg recycled
c_recycle = 40.0

#effective MRF cost per Mg of incoming waste:
#base tipping + recycling cost applied to the recycled fraction
c_M_eff = c_M + c_recycle * recycle_frac

#fixed daily operating costs (dollars/day)
F_L = 2000.0
F_M = 1500.0
F_W = 2500.0

#facility capacities (Mg/day of incoming waste)
cap_L = 200.0
cap_M = 350.0
cap_W = 210.0

#model

model = Model(HiGHS.Optimizer)

#normal variables: Mg/day of waste from each city to each facility
@variable(model, x_L[i in cities] >= 0)   # to landfill
@variable(model, x_M[i in cities] >= 0)   # to MRF
@variable(model, x_W[i in cities] >= 0)   # to WTE

#binary variables: whether each facility is opened
@variable(model, y_L, Bin)
@variable(model, y_M, Bin)
@variable(model, y_W, Bin)

#objective = minimize total cost (transport + tipping + recycling + fixed)
@objective(model, Min,
    sum(
        # transport cost
        c_trans * (d_L[i] * x_L[i] + d_M[i] * x_M[i] + d_W[i] * x_W[i])
        # disposal / processing costs
        + c_L * x_L[i]
        + c_M_eff * x_M[i]
        + c_W * x_W[i]
        for i in cities
    )
    + F_L * y_L + F_M * y_M + F_W * y_W
)

#constraints

#mass balance:
@constraint(model, [i in cities],
    x_L[i] + x_M[i] + x_W[i] == Q[i]
)

#landfill capacity: receives direct landfill waste + WTE ash
#WTE ash = ash_frac * (sum of waste going to WTE)
@constraint(model,
    sum(x_L[i] for i in cities) + ash_frac * sum(x_W[i] for i in cities) <= cap_L * y_L
)

#MRF and WTE capacity constraints (only incoming waste)
@constraint(model,
    sum(x_M[i] for i in cities) <= cap_M * y_M
)
@constraint(model,
    sum(x_W[i] for i in cities) <= cap_W * y_W
)

#solve optimization problem

optimize!(model)

println("\nTermination status: ", termination_status(model))
println("Optimal objective value (cost per day): ",
        round(objective_value(model); digits = 3))

println("\nFlows to landfill (x_L):")
for i in cities
    println("  City ", i, ": ", round(value(x_L[i]); digits = 3), " Mg/day")
end

println("\nFlows to MRF (x_M):")
for i in cities
    println("  City ", i, ": ", round(value(x_M[i]); digits = 3), " Mg/day")
end

println("\nFlows to WTE (x_W):")
for i in cities
    println("  City ", i, ": ", round(value(x_W[i]); digits = 3), " Mg/day")
end

println("\nFacility use:")
println("  Landfill open (y_L) = ", value(y_L))
println("  MRF open     (y_M) = ", value(y_M))
println("  WTE open     (y_W) = ", value(y_W))


Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 6 rows; 12 cols; 24 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [2e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [9e+01, 1e+02]
Presolving model
6 rows, 12 cols, 24 nonzeros  0s
6 rows, 12 cols, 24 nonzeros  0s
Presolve reductions: rows 6(-0); columns 12(-0); nonzeros 24(-0) - Not reduced

Solving MIP model with:
   6 rows
   12 cols (3 binary, 0 integer, 0 implied int., 9 continuous, 0 domain fixed)
   24 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |          

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

## References

List any external references consulted, including classmates.